# Loss Tails Above a Threshold

## Defining public losses within a fixed sample

For a positive index level \(P_t\), the daily loss in percentage points is

$$
L_t=-100\log\!\left(\frac{P_t}{P_{t-1}}\right).
$$

Here, \(P_t\) is the public NASDAQ Composite level on date \(t\), \(P_{t-1}\) is the preceding valid level, and \(L_t\) is positive when the index falls. The design sample ends on 2014-12-31. The held-out sample begins with the next available return, so no held-out loss enters the threshold or the frozen tail fit.

In [1]:
from pathlib import Path
import sys
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import HTML, display as _display

def display(value):
    module = value.__class__.__module__
    if module.startswith('pandas') and hasattr(value, '_repr_html_'):
        html = value._repr_html_()
        return _display(HTML('<div style="display:flex;justify-content:center;">' + html + '</div>'))
    return _display(value)
from scipy.stats import genpareto, norm
BOOK_ROOT = Path.cwd().parents[1]
if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))
from book_support import ensure_dataset
import RiskLabAI
from RiskLabAI.backtest.backtest_overfitting_simulation import expected_shortfall
from RiskLabAI.backtest.robust_statistics import conditional_expected_drawdown
from RiskLabAI.backtest.tail_risk import fit_pot_tail_risk, generalized_pareto_tail_risk, quantile_exception_tests
assert RiskLabAI.__version__ == '3.2.0'
DATA_PATH = ensure_dataset(BOOK_ROOT, 'fred_nasdaq')
FIGURES = BOOK_ROOT / 'outputs' / 'figures' / '21_tail_risk'
TABLES = BOOK_ROOT / 'outputs' / 'tables' / '21_tail_risk'
FIGURE_DATA = BOOK_ROOT / 'outputs' / 'figure_data' / '21_tail_risk'
for directory in [FIGURES, TABLES, FIGURE_DATA]:
    directory.mkdir(parents=True, exist_ok=True)
PALETTE = {'background': '#111827', 'axes': '#111827', 'text': '#E5E7EB', 'cyan': '#38BDF8', 'amber': '#FBBF24', 'green': '#34D399', 'rose': '#FB7185', 'violet': '#C084FC', 'grid': '#374151'}
plt.rcParams.update({'font.family': 'DejaVu Serif', 'font.size': 11.5, 'figure.facecolor': PALETTE['background'], 'axes.facecolor': PALETTE['axes'], 'axes.edgecolor': PALETTE['text'], 'axes.labelcolor': PALETTE['text'], 'axes.titlecolor': PALETTE['text'], 'xtick.color': PALETTE['text'], 'ytick.color': PALETTE['text'], 'text.color': PALETTE['text'], 'grid.color': PALETTE['grid'], 'grid.linestyle': ':', 'legend.facecolor': PALETTE['axes'], 'legend.edgecolor': PALETTE['grid'], 'savefig.facecolor': PALETTE['background'], 'savefig.transparent': False, 'savefig.dpi': 300})

def rendered_value(value):
    if pd.isna(value):
        return ''
    if isinstance(value, (bool, np.bool_)):
        return 'Pass' if bool(value) else 'Fail'
    if isinstance(value, (pd.Timestamp, np.datetime64)):
        return pd.Timestamp(value).date().isoformat()
    if isinstance(value, (int, np.integer)):
        return f'{int(value):,}'
    if isinstance(value, (float, np.floating)):
        number = float(value)
        if np.isposinf(number):
            return 'Infinite'
        if np.isneginf(number):
            return 'Negative infinite'
        if number == 0:
            return '0'
        if number.is_integer():
            return f'{int(number):,}'
        magnitude = abs(number)
        if magnitude < 0.0001 or magnitude >= 1000000:
            return f'{number:.6g}'
        return f'{number:.6f}'.rstrip('0').rstrip('.')
    return str(value)

def rendered_frame(frame):
    return frame.apply(lambda column: column.map(rendered_value))

def save_table(frame, stem, caption, label, fit=False):
    shown = rendered_frame(frame)
    shown.to_csv(TABLES / f'{stem}.csv', index=False)
    latex = shown.to_latex(index=False, escape=True, caption=caption, label=label, position='htbp')
    if fit:
        latex = latex.replace('\\begin{tabular}', '\\begin{adjustbox}{max width=\\textwidth}\\begin{tabular}', 1).replace('\\end{tabular}', '\\end{tabular}\\end{adjustbox}', 1)
    (TABLES / f'{stem}.tex').write_text(latex, encoding='utf-8')
    display(shown)
    return shown

def finish_axis(axis, grid=True):
    axis.spines['top'].set_visible(False)
    axis.spines['right'].set_visible(False)
    if grid:
        axis.grid(True, alpha=0.65)
    axis.set_axisbelow(True)

def save_figure(figure, stem):
    figure.savefig(FIGURES / f'{stem}.png', dpi=300, bbox_inches='tight', facecolor=PALETTE['background'], transparent=False)
    figure.savefig(FIGURES / f'{stem}.pdf', bbox_inches='tight', facecolor=PALETTE['background'], transparent=False)

def load_public_data():
    assert DATA_PATH.is_file()
    raw = pd.read_csv(DATA_PATH)
    raw['Date'] = pd.to_datetime(raw['observation_date'])
    raw['NASDAQ Composite'] = pd.to_numeric(raw['NASDAQCOM'], errors='coerce')
    data = raw.dropna(subset=['NASDAQ Composite']).sort_values('Date').reset_index(drop=True)
    data['Daily log return (%)'] = 100 * np.log(data['NASDAQ Composite'] / data['NASDAQ Composite'].shift(1))
    data['Loss (%)'] = -data['Daily log return (%)']
    data['Simple return'] = np.expm1(data['Daily log return (%)'] / 100)
    return data

def split_samples(data):
    returns = data.dropna(subset=['Loss (%)']).copy()
    design = returns[returns['Date'] <= pd.Timestamp('2014-12-31')].copy()
    held = returns[returns['Date'] >= pd.Timestamp('2015-01-01')].copy()
    assert len(design) == 5287
    assert len(held) == 2767
    return (returns, design, held)

def empirical_upper_tail_mean(losses, probability):
    values = np.sort(np.asarray(losses, dtype=float))
    count = (1 - probability) * len(values)
    full = int(np.floor(count))
    total = values[len(values) - full:].sum() if full > 0 else 0.0
    fraction = count - full
    if fraction > 0:
        total += fraction * values[len(values) - full - 1]
    return float(total / count)

def rolling_max_drawdowns(simple_returns, horizon):
    wealth = np.cumprod(1 + np.asarray(simple_returns, dtype=float))
    if len(wealth) < horizon or horizon < 2:
        drawdown = 1 - wealth / np.maximum.accumulate(wealth)
        return np.array([drawdown.max() if len(drawdown) else 0.0])
    windows = np.lib.stride_tricks.sliding_window_view(wealth, horizon)
    high_water = np.maximum.accumulate(windows, axis=1)
    return (1 - windows / high_water).max(axis=1)

def maximum_drawdown(simple_returns):
    wealth = np.cumprod(1 + np.asarray(simple_returns, dtype=float))
    drawdown = 1 - wealth / np.maximum.accumulate(wealth)
    return float(drawdown.max())

def moving_block_sample(values, length, block, rng):
    array = np.asarray(values, dtype=float)
    starts = rng.integers(0, len(array) - block + 1, size=int(np.ceil(length / block)))
    return np.concatenate([array[start:start + block] for start in starts])[:length]

def annual_pot_forecast(all_returns, held, probability):
    forecasts = pd.Series(index=held.index, dtype=float)
    records = []
    for year in sorted(held['Date'].dt.year.unique()):
        start = pd.Timestamp(f'{year}-01-01')
        history = all_returns[all_returns['Date'] < start]
        threshold = float(history['Loss (%)'].quantile(0.95))
        fit = fit_pot_tail_risk(history['Loss (%)'].to_numpy(), threshold=threshold, probability=probability)
        mask = held['Date'].dt.year == year
        forecasts.loc[held.index[mask]] = fit['value_at_risk']
        records.append({'Forecast year': int(year), 'History through': history['Date'].max(), 'History observations': int(len(history)), 'Threshold (%)': threshold, 'Excesses': int(fit['exceedance_count']), 'Shape': fit['shape'], 'Scale (%)': fit['scale'], 'Forecast VaR (%)': fit['value_at_risk'], 'Forecast ES (%)': fit['expected_shortfall']})
    assert forecasts.notna().all()
    return (forecasts, pd.DataFrame(records))
data = load_public_data()
returns, design, held = split_samples(data)
provenance = pd.DataFrame([{'Item': 'Public series', 'Value': 'NASDAQ Composite, FRED NASDAQCOM'}, {'Item': 'Stored file', 'Value': DATA_PATH.name}, {'Item': 'Valid index observations', 'Value': int(data['NASDAQ Composite'].notna().sum())}, {'Item': 'Daily returns', 'Value': int(len(returns))}, {'Item': 'Download during execution', 'Value': 'None'}, {'Item': 'Simulated market observations', 'Value': 'None'}])
save_table(provenance, 'data_provenance', 'Public data provenance for Chapter 22.', 'tab:chapter22-data-provenance', fit=True)
periods = pd.DataFrame([{'Sample': 'Design', 'First return': design['Date'].min(), 'Last return': design['Date'].max(), 'Returns': len(design)}, {'Sample': 'Held out', 'First return': held['Date'].min(), 'Last return': held['Date'].max(), 'Returns': len(held)}, {'Sample': 'Complete', 'First return': returns['Date'].min(), 'Last return': returns['Date'].max(), 'Returns': len(returns)}])
save_table(periods, 'sample_periods', 'Chronological sample boundaries.', 'tab:chapter22-sample-periods')
notation = pd.DataFrame([{'Symbol': 'P(t)', 'Description': 'Index level on date t'}, {'Symbol': 'L(t)', 'Description': 'Daily percentage loss'}, {'Symbol': 'u', 'Description': 'Loss threshold fixed from the design sample'}, {'Symbol': 'Y', 'Description': 'Positive excess L minus u when L exceeds u'}, {'Symbol': 'xi', 'Description': 'Generalized Pareto shape parameter'}, {'Symbol': 'beta', 'Description': 'Generalized Pareto scale parameter'}, {'Symbol': 'p(u)', 'Description': 'Probability that a loss exceeds u'}])
save_table(notation, 'tail_notation', 'Notation for the threshold tail model.', 'tab:chapter22-tail-notation')
summary_rows = []
for name, frame in [('Design', design), ('Held out', held), ('Complete', returns)]:
    losses = frame['Loss (%)']
    summary_rows.append({'Sample': name, 'Observations': len(losses), 'Mean loss (%)': losses.mean(), 'Standard deviation (%)': losses.std(ddof=1), '95th percentile (%)': losses.quantile(0.95), '99th percentile (%)': losses.quantile(0.99), 'Largest loss (%)': losses.max()})
loss_summary = pd.DataFrame(summary_rows)
save_table(loss_summary, 'loss_summary_by_period', 'Loss summaries for the fixed samples.', 'tab:chapter22-loss-summary', fit=True)
history = data[['Date', 'NASDAQ Composite', 'Loss (%)']].copy()
history.to_csv(FIGURE_DATA / 'public_price_loss_history.csv', index=False)
figure, axes = plt.subplots(2, 1, figsize=(11.2, 7.2), sharex=True, constrained_layout=True)
axes[0].plot(history['Date'], history['NASDAQ Composite'], color=PALETTE['cyan'], linewidth=1.2)
axes[0].set_yscale('log')
axes[0].set_ylabel('Index level')
axes[0].set_title('Public NASDAQ Composite and Daily Percentage Losses')
axes[1].plot(history['Date'], history['Loss (%)'], color=PALETTE['amber'], linewidth=0.58)
axes[1].axvline(pd.Timestamp('2014-12-31'), color=PALETTE['violet'], linestyle='--', linewidth=1.4, label='Design sample ends')
axes[1].set_ylabel('Daily loss (%)')
axes[1].set_xlabel('Date')
axes[1].legend()
for axis in axes:
    finish_axis(axis)
save_figure(figure, 'public_price_loss_history')
plt.close(figure)
assert design['Date'].max() < held['Date'].min()
assert len(design) + len(held) == len(returns)


,Item,Value
0,Public series,"NASDAQ Composite, FRED NASDAQCOM"
1,Stored file,NASDAQCOM_1994-01-01_2025-12-31.csv
2,Valid index observations,"8,055"
3,Daily returns,"8,054"
4,Download during execution,None
5,Simulated market observations,None


,Sample,First return,Last return,Returns
0,Design,1994-01-04,2014-12-31,"5,287"
1,Held out,2015-01-02,2025-12-31,"2,767"
2,Complete,1994-01-04,2025-12-31,"8,054"


,Symbol,Description
0,P(t),Index level on date t
1,L(t),Daily percentage loss
2,u,Loss threshold fixed from the design sample
3,Y,Positive excess L minus u when L exceeds u
4,xi,Generalized Pareto shape parameter
5,beta,Generalized Pareto scale parameter
6,p(u),Probability that a loss exceeds u


,Sample,Observations,Mean loss (%),Standard deviation (%),95th percentile (%),99th percentile (%),Largest loss (%)
0,Design,"5,287",-0.03434,1.599634,2.614336,4.4378,10.168411
1,Held out,"2,767",-0.05749,1.364332,2.219784,3.884087,13.149154
2,Complete,"8,054",-0.042294,1.522854,2.505396,4.249808,13.149154


## Modeling threshold excesses with the generalized Pareto law

For a threshold \(u\), define the excess \(Y=L-u\) only when \(L>u\). The generalized Pareto survival law is

$$
\Pr(Y>y\mid L>u)=\left(1+\frac{\xi y}{\beta}\right)^{-1/\xi},
$$

on values of \(y\geq 0\) for which \(1+\xi y/\beta>0\). The scale parameter \(\beta>0\) sets the size of excesses. The shape parameter \(\xi\) controls how slowly the tail declines. When \(\xi=0\), the continuous limit is the exponential survival law \(\exp(-y/\beta)\).

In [2]:
data = load_public_data()
returns, design, held = split_samples(data)
design_losses = design["Loss (%)"].to_numpy()
threshold_levels = [0.90, 0.925, 0.95, 0.975]
threshold_rows = []
for level in threshold_levels:
    threshold = float(np.quantile(design_losses, level))
    excesses = design_losses[design_losses > threshold] - threshold
    threshold_rows.append({
        "Design percentile": level,
        "Threshold (%)": threshold,
        "Excesses": len(excesses),
        "Exceedance proportion": len(excesses) / len(design_losses),
        "Mean excess (%)": excesses.mean(),
        "Largest excess (%)": excesses.max(),
    })
threshold_counts = pd.DataFrame(threshold_rows)
save_table(threshold_counts, "threshold_counts", "Design-sample thresholds and excess counts.", "tab:chapter22-threshold-counts", fit=True)

primary_threshold = float(np.quantile(design_losses, 0.95))
primary_fit = fit_pot_tail_risk(design_losses, threshold=primary_threshold, probability=0.99)
primary_parameters = pd.DataFrame([
    {"Quantity": "Threshold percentile", "Value": 0.95},
    {"Quantity": "Threshold (%)", "Value": primary_fit["threshold"]},
    {"Quantity": "Design observations", "Value": primary_fit["sample_count"]},
    {"Quantity": "Positive excesses", "Value": primary_fit["exceedance_count"]},
    {"Quantity": "Exceedance probability", "Value": primary_fit["exceedance_probability"]},
    {"Quantity": "Shape", "Value": primary_fit["shape"]},
    {"Quantity": "Scale (%)", "Value": primary_fit["scale"]},
])
save_table(primary_parameters, "pot_primary_parameters", "Primary generalized Pareto fit from the design sample.", "tab:chapter22-pot-primary")

survival_grid = np.linspace(max(0, np.quantile(design_losses, 0.75)), design_losses.max(), 300)
survival = pd.DataFrame({
    "Loss (%)": survival_grid,
    "Empirical survival probability": [np.mean(design_losses > value) for value in survival_grid],
})
survival.to_csv(FIGURE_DATA / "empirical_loss_survival.csv", index=False)
figure, axis = plt.subplots(figsize=(10.6, 5.8), constrained_layout=True)
axis.semilogy(survival["Loss (%)"], survival["Empirical survival probability"], color=PALETTE["cyan"], linewidth=1.8)
axis.axvline(primary_threshold, color=PALETTE["amber"], linestyle="--", linewidth=1.5, label="95th percentile threshold")
axis.set_xlabel("Daily loss (%)")
axis.set_ylabel("Probability of a larger loss")
axis.set_title("Empirical Upper Tail in the Design Sample")
axis.legend()
finish_axis(axis)
save_figure(figure, "empirical_loss_survival")
plt.close(figure)

timeline = design[["Date", "Loss (%)"]].copy()
timeline["Exceeds threshold"] = timeline["Loss (%)"] > primary_threshold
timeline["Threshold (%)"] = primary_threshold
timeline.to_csv(FIGURE_DATA / "threshold_exceedance_timeline.csv", index=False)
figure, axis = plt.subplots(figsize=(11.0, 5.4), constrained_layout=True)
axis.plot(timeline["Date"], timeline["Loss (%)"], color=PALETTE["text"], linewidth=0.55, alpha=0.72)
marked = timeline["Exceeds threshold"]
axis.scatter(timeline.loc[marked, "Date"], timeline.loc[marked, "Loss (%)"], color=PALETTE["rose"], s=11, alpha=0.8, label="Threshold excess")
axis.axhline(primary_threshold, color=PALETTE["amber"], linestyle="--", linewidth=1.5, label="Fixed threshold")
axis.set_xlabel("Date")
axis.set_ylabel("Daily loss (%)")
axis.set_title("Design-Sample Losses Above the Fixed Threshold")
axis.legend()
finish_axis(axis)
save_figure(figure, "threshold_exceedance_timeline")
plt.close(figure)

assert primary_fit["exceedance_count"] == 265
assert primary_fit["scale"] > 0
assert np.all(1 + primary_fit["shape"] * (design_losses[design_losses > primary_threshold] - primary_threshold) / primary_fit["scale"] > 0)

,Design percentile,Threshold (%),Excesses,Exceedance proportion,Mean excess (%),Largest excess (%)
0,0.9,1.831137,529,0.100057,1.170777,8.337273
1,0.925,2.174873,397,0.07509,1.15877,7.993538
2,0.95,2.614336,265,0.050123,1.196085,7.554075
3,0.975,3.303977,133,0.025156,1.38023,6.864434


,Quantity,Value
0,Threshold percentile,0.95
1,Threshold (%),2.614336
2,Design observations,"5,287"
3,Positive excesses,265
4,Exceedance probability,0.050123
5,Shape,0.135989
6,Scale (%),1.036079


## Testing generalized Pareto fit across thresholds

For positive excesses \(y_1,\ldots,y_k\), the log likelihood is

$$
\ell(\xi,\beta)=-k\log\beta-\left(1+\frac{1}{\xi}\right)\sum_{i=1}^{k}\log\!\left(1+\frac{\xi y_i}{\beta}\right),
$$

when \(\xi\neq 0\) and every support term is positive. Here, \(k\) is the number of excesses, \(y_i\) is excess \(i\), \(\xi\) is the shape, and \(\beta\) is the positive scale. The fitting calculation fixes the excess location at zero and chooses the shape and scale numerically. A numerical maximum is a fitted solution, not a proof of a unique global maximum.

The mean excess function at a candidate threshold \(u\) is

$$
e(u)=\mathbb{E}[L-u\mid L>u].
$$

The value \(e(u)\) is the average distance above \(u\), using only losses that exceed \(u\). A nearly linear pattern over a range of high thresholds is one useful diagnostic, but it does not determine a unique threshold.

In [3]:
data = load_public_data()
returns, design, held = split_samples(data)
losses = design["Loss (%)"].to_numpy()
sensitivity_rows = []
for level in [0.90, 0.925, 0.95, 0.975]:
    threshold = float(np.quantile(losses, level))
    fit = fit_pot_tail_risk(losses, threshold=threshold, probability=0.99)
    excesses = losses[losses > threshold] - threshold
    support_minimum = float(np.min(1 + fit["shape"] * excesses / fit["scale"]))
    log_likelihood = float(genpareto.logpdf(excesses, fit["shape"], loc=0, scale=fit["scale"]).sum())
    sensitivity_rows.append({
        "Threshold percentile": level,
        "Threshold (%)": threshold,
        "Excesses": fit["exceedance_count"],
        "Shape": fit["shape"],
        "Scale (%)": fit["scale"],
        "99% VaR (%)": fit["value_at_risk"],
        "99% ES (%)": fit["expected_shortfall"],
        "Smallest support term": support_minimum,
        "Log likelihood": log_likelihood,
    })
sensitivity = pd.DataFrame(sensitivity_rows)
save_table(sensitivity, "pot_threshold_sensitivity", "Generalized Pareto sensitivity to the design-sample threshold.", "tab:chapter22-pot-sensitivity", fit=True)

parameter_data = sensitivity[["Threshold percentile", "Shape", "Scale (%)", "99% VaR (%)", "99% ES (%)"]].copy()
parameter_data.to_csv(FIGURE_DATA / "threshold_parameter_stability.csv", index=False)
figure, axes = plt.subplots(1, 2, figsize=(10.8, 4.8), constrained_layout=True)
axes[0].plot(parameter_data["Threshold percentile"], parameter_data["Shape"], color=PALETTE["cyan"], marker="o", linewidth=1.8)
axes[0].axhline(0, color=PALETTE["grid"], linewidth=1)
axes[0].set_xlabel("Design-sample percentile")
axes[0].set_ylabel("Fitted shape")
axes[0].set_title("Shape Estimate")
axes[1].plot(parameter_data["Threshold percentile"], parameter_data["Scale (%)"], color=PALETTE["amber"], marker="s", linewidth=1.8)
axes[1].set_xlabel("Design-sample percentile")
axes[1].set_ylabel("Fitted scale (%)")
axes[1].set_title("Scale Estimate")
for axis in axes:
    finish_axis(axis)
save_figure(figure, "threshold_parameter_stability")
plt.close(figure)

mean_excess_rows = []
for level in np.arange(0.80, 0.986, 0.005):
    threshold = float(np.quantile(losses, level))
    excesses = losses[losses > threshold] - threshold
    mean_excess_rows.append({"Threshold percentile": level, "Threshold (%)": threshold, "Mean excess (%)": excesses.mean(), "Excesses": len(excesses)})
mean_excess = pd.DataFrame(mean_excess_rows)
mean_excess.to_csv(FIGURE_DATA / "mean_excess_diagnostic.csv", index=False)
figure, axis = plt.subplots(figsize=(10.4, 5.4), constrained_layout=True)
axis.plot(mean_excess["Threshold (%)"], mean_excess["Mean excess (%)"], color=PALETTE["green"], linewidth=1.8)
axis.axvline(float(np.quantile(losses, 0.95)), color=PALETTE["amber"], linestyle="--", linewidth=1.4, label="Primary threshold")
axis.set_xlabel("Threshold (%)")
axis.set_ylabel("Mean excess (%)")
axis.set_title("Mean Excess Across Candidate Thresholds")
axis.legend()
finish_axis(axis)
save_figure(figure, "mean_excess_diagnostic")
plt.close(figure)

primary = sensitivity.loc[np.isclose(sensitivity["Threshold percentile"], 0.95)].iloc[0]
primary_threshold = float(primary["Threshold (%)"])
excesses = np.sort(losses[losses > primary_threshold] - primary_threshold)
probabilities = (np.arange(1, len(excesses) + 1) - 0.5) / len(excesses)
theoretical = genpareto.ppf(probabilities, float(primary["Shape"]), loc=0, scale=float(primary["Scale (%)"]))
qq = pd.DataFrame({"Fitted quantile (%)": theoretical, "Observed excess (%)": excesses})
qq.to_csv(FIGURE_DATA / "generalized_pareto_qq.csv", index=False)
figure, axis = plt.subplots(figsize=(6.6, 6.0), constrained_layout=True)
axis.scatter(qq["Fitted quantile (%)"], qq["Observed excess (%)"], color=PALETTE["cyan"], s=17, alpha=0.72)
limit = max(qq.max())
axis.plot([0, limit], [0, limit], color=PALETTE["amber"], linestyle="--", linewidth=1.4, label="Equal quantiles")
axis.set_xlabel("Fitted excess quantile (%)")
axis.set_ylabel("Observed excess quantile (%)")
axis.set_title("Generalized Pareto Quantile Check")
axis.legend()
finish_axis(axis)
save_figure(figure, "generalized_pareto_qq")
plt.close(figure)

checks = pd.DataFrame([
    {"Check": "Threshold uses design sample only", "Value": design["Date"].max() < held["Date"].min(), "Required condition": "Pass"},
    {"Check": "Positive distinct excesses", "Value": len(excesses) >= 2 and np.ptp(excesses) > 0, "Required condition": "Pass"},
    {"Check": "Positive fitted scale", "Value": float(primary["Scale (%)"]) > 0, "Required condition": "Pass"},
    {"Check": "Support terms positive", "Value": float(primary["Smallest support term"]) > 0, "Required condition": "Pass"},
    {"Check": "Finite fitted log likelihood", "Value": np.isfinite(float(primary["Log likelihood"])), "Required condition": "Pass"},
])
save_table(checks, "support_likelihood_checks", "Support and likelihood checks for the primary fit.", "tab:chapter22-support-checks", fit=True)

limits = pd.DataFrame([
    {"Question": "What is fitted?", "Answer": "Positive excesses above a supplied threshold"},
    {"Question": "What is not fitted?", "Answer": "The full loss distribution"},
    {"Question": "How is the threshold chosen?", "Answer": "From the design sample before held-out evaluation"},
    {"Question": "What does a diagnostic establish?", "Answer": "Finite-sample fit behavior, not truth"},
    {"Question": "What can dependence change?", "Answer": "Uncertainty and finite-sample stability"},
])
save_table(limits, "tail_model_limits", "Interpretation limits for the threshold tail model.", "tab:chapter22-tail-limits", fit=True)

,Threshold percentile,Threshold (%),Excesses,Shape,Scale (%),99% VaR (%),99% ES (%),Smallest support term,Log likelihood
0,0.9,1.831137,529,0.066155,1.093188,4.55085,5.91415,1.000033,-611.179281
1,0.925,2.174873,397,0.112127,1.029798,4.504446,5.958488,1.000139,-453.16476
2,0.95,2.614336,265,0.135989,1.036079,4.481563,5.974601,1.000056,-310.425806
3,0.975,3.303977,133,0.032208,1.335945,4.554896,5.976933,1.000019,-175.805988


,Check,Value,Required condition
0,Threshold uses design sample only,Pass,Pass
1,Positive distinct excesses,Pass,Pass
2,Positive fitted scale,Pass,Pass
3,Support terms positive,Pass,Pass
4,Finite fitted log likelihood,Pass,Pass


,Question,Answer
0,What is fitted?,Positive excesses above a supplied threshold
1,What is not fitted?,The full loss distribution
2,How is the threshold chosen?,From the design sample before held-out evaluation
3,What does a diagnostic establish?,"Finite-sample fit behavior, not truth"
4,What can dependence change?,Uncertainty and finite-sample stability


,Question,Answer
0,What is fitted?,Positive excesses above a supplied threshold
1,What is not fitted?,The full loss distribution
2,How is the threshold chosen?,From the design sample before held-out evaluation
3,What does a diagnostic establish?,"Finite-sample fit behavior, not truth"
4,What can dependence change?,Uncertainty and finite-sample stability


## Extrapolating high loss quantiles

If \(p_u=\Pr(L>u)\), the fitted probability of a loss above \(x>u\) is

$$
\widehat{\Pr}(L>x)=\widehat p_u\left(1+\frac{\widehat\xi(x-u)}{\widehat\beta}\right)^{-1/\widehat\xi}.
$$

Here, \(\widehat p_u=k/n\) is the design-sample fraction above the threshold, \(n\) is the number of design losses, and \(k\) is the excess count. The fitted shape and scale are \(\widehat\xi\) and \(\widehat\beta\). For a target probability \(p>1-\widehat p_u\), inversion gives

$$
\widehat q_p=u+\frac{\widehat\beta}{\widehat\xi}\left[\left(\frac{1-p}{\widehat p_u}\right)^{-\widehat\xi}-1\right],
$$

with the exponential limit used when \(\widehat\xi=0\). The quantity \(\widehat q_p\) is the fitted loss quantile whose exceedance probability is \(1-p\).

In [4]:
data = load_public_data()
returns, design, held = split_samples(data)
losses = design["Loss (%)"].to_numpy()
threshold = float(np.quantile(losses, 0.95))
fit = fit_pot_tail_risk(losses, threshold=threshold, probability=0.99)
risk_rows = []
for probability in [0.975, 0.99, 0.995]:
    risk = generalized_pareto_tail_risk(threshold, fit["exceedance_probability"], fit["shape"], fit["scale"], probability)
    risk_rows.append({"Quantile probability": probability, "Tail probability": 1 - probability, "Value at risk (%)": risk["value_at_risk"], "Expected shortfall (%)": risk["expected_shortfall"]})
risk_estimates = pd.DataFrame(risk_rows)
save_table(risk_estimates, "pot_risk_estimates", "Fitted tail risk at three probability levels.", "tab:chapter22-pot-risk")

x_grid = np.linspace(threshold, max(threshold + 0.01, losses.max()), 240)
empirical = np.array([np.mean(losses > value) for value in x_grid])
fitted = fit["exceedance_probability"] * np.power(1 + fit["shape"] * (x_grid - threshold) / fit["scale"], -1 / fit["shape"])
tail_curve = pd.DataFrame({"Loss (%)": x_grid, "Empirical probability": empirical, "Fitted probability": fitted})
tail_curve.to_csv(FIGURE_DATA / "fitted_empirical_tail_probability.csv", index=False)
figure, axis = plt.subplots(figsize=(10.5, 5.7), constrained_layout=True)
axis.semilogy(tail_curve["Loss (%)"], tail_curve["Empirical probability"], color=PALETTE["cyan"], linewidth=1.7, label="Empirical")
axis.semilogy(tail_curve["Loss (%)"], tail_curve["Fitted probability"], color=PALETTE["amber"], linewidth=1.8, label="Generalized Pareto fit")
axis.set_xlabel("Daily loss (%)")
axis.set_ylabel("Probability of a larger loss")
axis.set_title("Empirical and Fitted Probabilities Above the Threshold")
axis.legend()
finish_axis(axis)
save_figure(figure, "fitted_empirical_tail_probability")
plt.close(figure)

manual = threshold + fit["scale"] / fit["shape"] * (((1 - 0.99) / fit["exceedance_probability"]) ** (-fit["shape"]) - 1)
assert np.isclose(manual, fit["value_at_risk"], rtol=1e-12, atol=1e-12)
assert all(risk_estimates["Expected shortfall (%)"] > risk_estimates["Value at risk (%)"])
assert np.all(fitted > 0)

,Quantile probability,Tail probability,Value at risk (%),Expected shortfall (%)
0,0.975,0.025,3.370224,4.688345
1,0.99,0.01,4.481563,5.974601
2,0.995,0.005,5.419222,7.05984
